# vbOCP — Mega sweep PODNN + GNN

Notebook separato da `run_pipeline_colab.ipynb`: qui si fanno sweep lunghi/costosi, non la pipeline principale. Riusa solo script gia' testati (`generate_snapshots.py`, `sweep_pod_nn.py`, `convert_to_gca_rom.py`, `train_gnn.py`, `evaluate_gnn.py`) - nessuna logica nuova, solo orchestrazione a piu' parametri.

**Contenuto:**
1. Setup (stesso pattern di `run_pipeline_colab.ipynb`)
2. Mega sweep PODNN: griglia `N_samples x N_modes` (veloce, minuti per punto)
3. Mega sweep GNN: 1D su `N_samples` a `bottleneck_dim` fisso (lento, ~40 min per punto - lista di default volutamente corta)
4. Aggregazione risultati + plot

Decisioni di questa sessione gia' incorporate in `sweep_pod_nn.py` (vedi handout): niente standardizzazione dei coefficienti in output (peggiorava l'errore di 2-4x), early stopping attivo durante il training.

## 0. Setup Colab (installazione + clone repo)

In [ ]:
GITHUB_URL = 'https://github.com/roccnroll/vbOCP.git'
REPO_DIR = 'vbOCP'

!pip install -q pypolydim gmsh
!cd vbOCP && git pull
import os
if not os.path.isdir(REPO_DIR):
    !git clone {GITHUB_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} gia' presente (skip clone) - se vuoi aggiornarla: !cd {REPO_DIR} && git pull")

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
import os

while not os.path.isdir('src') and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir('..')
assert os.path.isdir('src'), "src/ non trovata: verifica dove e' montata la repo vbOCP"

REPO_ROOT     = os.getcwd()
CONFIG_PATH   = os.path.join('configs', 'test1.yaml')
SNAPSHOTS_DIR = os.path.join('data', 'snapshots')
GNN_DIR       = os.path.join('data', 'gnn')
GNN_MODELS_DIR = os.path.join(GNN_DIR, 'models')

os.makedirs(SNAPSHOTS_DIR, exist_ok=True)
os.makedirs(GNN_DIR, exist_ok=True)

print('REPO_ROOT    :', REPO_ROOT)
print('CONFIG_PATH  :', CONFIG_PATH)
print('SNAPSHOTS_DIR:', SNAPSHOTS_DIR)

In [ ]:
# test set FISSO per tutta la mega-sweep (sia PODNN che GNN) - stesso seed diverso da quello di
# training (999 vs 42) cosi' non si sovrappone a nessuno dei training set generati sotto, e
# tutti i confronti (a N_samples/N_modes diversi) sono fatti sugli stessi identici campioni
TEST_SNAPSHOTS_FIXED = os.path.join(SNAPSHOTS_DIR, 'test1_test150.npz')
if not os.path.exists(TEST_SNAPSHOTS_FIXED):
    !python -m src.full_order.generate_snapshots \
        --config {CONFIG_PATH} --n-samples 150 --seed 999 \
        --output {TEST_SNAPSHOTS_FIXED}
else:
    print(f"{TEST_SNAPSHOTS_FIXED} gia' presente (skip)")

def gen_train_snapshots_if_missing(n_samples, seed=42):
    path = os.path.join(SNAPSHOTS_DIR, f'test1_{n_samples}.npz')
    if os.path.exists(path):
        print(f"{path} gia' presente (skip)")
        return path
    !python -m src.full_order.generate_snapshots \
        --config {CONFIG_PATH} --n-samples {n_samples} --seed {seed} \
        --output {path}
    return path

## 1. Mega sweep PODNN: `N_samples` x `N_modes`

Per ogni valore di `N_samples`: genera (se manca) gli snapshot di training, poi lancia `sweep_pod_nn.py` con la lista di `N_modes` - lo script ricostruisce base POD + rete per ogni valore, valuta su `TEST_SNAPSHOTS_FIXED`, con early stopping attivo (vedi handout).

**Nota sul cap di N_modes**: l'autodecomposizione della matrice di correlazione (M x M, M = N_samples) ha al massimo M autovalori/autovettori non nulli - un `N_modes >= N_samples` non ha senso. Il loop scarta automaticamente i valori non validi per ogni N_samples.

Modifica le due liste sotto per cambiare la scala dello sweep - quelle di default sono un punto di partenza ragionevole (griglia 5x7 = 35 run, ciascuno pochi minuti).

In [ ]:
POD_N_SAMPLES_LIST = [150, 300, 450, 600, 750]
POD_N_MODES_LIST = [2, 5, 8, 11, 14, 17, 20]
POD_EPOCHS = 30000
POD_SWEEP_DIR = os.path.join(SNAPSHOTS_DIR, 'mega_sweep_podnn')
os.makedirs(POD_SWEEP_DIR, exist_ok=True)

print(f"Griglia: {len(POD_N_SAMPLES_LIST)} x {len(POD_N_MODES_LIST)} (al netto dei tagli per N_modes >= N_samples)")

In [ ]:
for n in POD_N_SAMPLES_LIST:
    train_path = gen_train_snapshots_if_missing(n)

    values_n = [v for v in POD_N_MODES_LIST if v < n]
    if not values_n:
        print(f"N_samples={n}: nessun N_modes valido (tutti >= {n}), skip\n")
        continue

    out_csv = os.path.join(POD_SWEEP_DIR, f'sweep_modes_N{n}.csv')
    if os.path.exists(out_csv):
        print(f"{out_csv} gia' presente (skip - cancella per rilanciare questo punto)\n")
        continue

    values_str = ','.join(str(v) for v in values_n)
    print(f"=== N_samples={n}  N_modes={values_n} ===")
    !python -m src.rom.sweep_pod_nn \
        --config {CONFIG_PATH} \
        --snapshots {train_path} \
        --test-snapshots {TEST_SNAPSHOTS_FIXED} \
        --sweep-param n_modes --values {values_str} \
        --epochs {POD_EPOCHS} \
        --output {out_csv}
    print()

### 1.1 Aggregazione risultati PODNN (curve errore vs N_modes, una per N_samples)

In [ ]:
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pod_results = {}  # (n_samples, n_modes) -> (err_y, err_p)
for n in POD_N_SAMPLES_LIST:
    out_csv = os.path.join(POD_SWEEP_DIR, f'sweep_modes_N{n}.csv')
    if not os.path.exists(out_csv):
        continue
    with open(out_csv) as f:
        for row in csv.DictReader(f):
            pod_results[(n, int(row['n_modes']))] = (float(row['err_y']), float(row['err_p']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, field_idx, title in zip(axes, [0, 1], ['Stato (y)', 'Aggiunto (p)']):
    for n in POD_N_SAMPLES_LIST:
        pts = sorted((m, v[field_idx]) for (ns, m), v in pod_results.items() if ns == n)
        if not pts:
            continue
        modes, errs = zip(*pts)
        ax.semilogy(modes, errs, 'o-', label=f'N_samples={n}')
    ax.set_xlabel('N_modes')
    ax.set_ylabel('errore relativo')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(POD_SWEEP_DIR, 'mega_sweep_podnn_summary.png')
plt.savefig(plot_path, dpi=90)
print(f"Plot salvato in {plot_path}")

from IPython.display import Image, display
display(Image(plot_path))

## 2. Mega sweep GNN: `N_samples` (bottleneck_dim fisso)

Un run GNN e' costoso (~40 min con `GNN_EPOCHS` di default) - la lista sotto e' volutamente corta. Riusa gli stessi iperparametri "noti buoni" gia' fissati in questa sessione (`ffn=200`, `in_channels=2`, `scaling_type=4`, `--comp 2` = y e p come due canali) - nessuna modifica allo scaler GNN (lasciato com'e', vedi handout).

### 2.0 Setup gca-rom

In [ ]:
GCA_ROM_PATH = 'gca-rom'
import importlib.util
if importlib.util.find_spec('torch_geometric') is None:
    !pip install -q torch_geometric

if not os.path.isdir(GCA_ROM_PATH):
    !git clone https://github.com/fpichi/gca-rom.git {GCA_ROM_PATH}
else:
    print(f"{GCA_ROM_PATH} gia' presente (skip clone)")

In [ ]:
GNN_N_SAMPLES_LIST = [50, 100, 150, 300]   # lista corta di proposito, vedi nota sopra sul costo
GNN_BOTTLENECK = 15                         # fisso - stesso default gia' usato nella pipeline principale
GNN_EPOCHS = 5000
GNN_SWEEP_DIR = os.path.join(GNN_DIR, 'mega_sweep')
os.makedirs(GNN_SWEEP_DIR, exist_ok=True)

def convert_if_missing(snapshots_path, output_path, n_comp, field=None):
    if os.path.exists(output_path):
        print(f"{output_path} gia' presente (skip)")
        return
    field_flag = f'--field {field}' if field is not None else ''
    !python -m src.gnn.convert_to_gca_rom \
        --config {CONFIG_PATH} \
        --snapshots {snapshots_path} \
        --n-comp {n_comp} \
        {field_flag} \
        --output {output_path}

# .mat di test FISSO (stesso per tutti i punti dello sweep) - condiviso con run_pipeline_colab.ipynb
# se gia' generato li' (stesso nome file, stesso TEST_SNAPSHOTS_FIXED)
TEST_YP_MAT = os.path.join(GNN_DIR, 'test_yp.mat')
convert_if_missing(TEST_SNAPSHOTS_FIXED, TEST_YP_MAT, n_comp=2)

In [ ]:
import json

def train_gnn_if_missing(train_mat, net_name):
    net_dir = os.path.join(GNN_MODELS_DIR, net_name)
    meta_path = os.path.join(net_dir, 'train_meta.json')
    target = {'epochs': GNN_EPOCHS, 'bottleneck_dim': GNN_BOTTLENECK}
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if all(meta.get(k) == v for k, v in target.items()):
            print(f"{meta_path} gia' presente con parametri identici (skip)")
            return net_dir
        print(f"{meta_path} presente ma con parametri diversi - cancella {net_dir} per riallenare")
        return net_dir
    !python -m src.gnn.train_gnn \
        --config {CONFIG_PATH} \
        --gca-rom-path {GCA_ROM_PATH} \
        --train-mat {train_mat} \
        --test-mat {TEST_YP_MAT} \
        --comp 2 \
        --net-name {net_name} \
        --net-dir {GNN_MODELS_DIR} \
        --bottleneck-dim {GNN_BOTTLENECK} \
        --epochs {GNN_EPOCHS}
    return net_dir

for n in GNN_N_SAMPLES_LIST:
    train_snap_path = gen_train_snapshots_if_missing(n)
    train_mat = os.path.join(GNN_SWEEP_DIR, f'train_yp_N{n}.mat')
    convert_if_missing(train_snap_path, train_mat, n_comp=2)

    net_name = f'mega_sweep_gnn_yp_N{n}'
    print(f"=== N_samples={n} ===")
    train_gnn_if_missing(train_mat, net_name)
    print()

### 2.1 Valutazione + aggregazione risultati GNN

In [ ]:
for n in GNN_N_SAMPLES_LIST:
    train_mat = os.path.join(GNN_SWEEP_DIR, f'train_yp_N{n}.mat')
    net_name = f'mega_sweep_gnn_yp_N{n}'
    net_dir = os.path.join(GNN_MODELS_DIR, net_name)
    out_csv = os.path.join(GNN_SWEEP_DIR, f'err_N{n}.csv')
    if os.path.exists(out_csv):
        print(f"{out_csv} gia' presente (skip)")
        continue
    print(f"=== valutazione N_samples={n} ===")
    !python -m src.gnn.evaluate_gnn \
        --config {CONFIG_PATH} \
        --gca-rom-path {GCA_ROM_PATH} \
        --train-mat {train_mat} \
        --test-mat {TEST_YP_MAT} \
        --net-dir {net_dir} \
        --save-csv {out_csv}
    print()

In [ ]:
import csv
import numpy as np

gnn_results = {}  # n -> (err_y_l2_mean, err_p_l2_mean)
for n in GNN_N_SAMPLES_LIST:
    out_csv = os.path.join(GNN_SWEEP_DIR, f'err_N{n}.csv')
    if not os.path.exists(out_csv):
        continue
    with open(out_csv) as f:
        rows = list(csv.DictReader(f))
    err_y = np.mean([float(r['err_y_l2']) for r in rows])
    err_p = np.mean([float(r['err_p_l2']) for r in rows])
    gnn_results[n] = (err_y, err_p)

ns = sorted(gnn_results)
fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(ns, [gnn_results[n][0] for n in ns], 'o-', label='Stato (y)')
ax.semilogy(ns, [gnn_results[n][1] for n in ns], 's-', label='Aggiunto (p)')
ax.set_xlabel('N_samples (training)')
ax.set_ylabel('errore relativo L2 (medio sul test set)')
ax.set_title(f'GNN - sweep N_samples (bottleneck_dim={GNN_BOTTLENECK})')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(GNN_SWEEP_DIR, 'mega_sweep_gnn_summary.png')
plt.savefig(plot_path, dpi=90)
print(f"Plot salvato in {plot_path}")
display(Image(plot_path))

## 3. Salvataggio risultati sulla repo (opzionale)

Solo i CSV/PNG di riepilogo (non i pesi GNN completi, che sarebbero troppi per una griglia - se serve tenerne uno specifico, copialo a mano in `data/gnn/models/` col pattern gia' usato da `run_pipeline_colab.ipynb`).

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
remote_url = f'https://{token}@github.com/roccnroll/vbOCP.git'

!git add {POD_SWEEP_DIR}/*.csv {POD_SWEEP_DIR}/*.png {GNN_SWEEP_DIR}/*.csv {GNN_SWEEP_DIR}/*.png
!git status --short
!git commit -m "Aggiunge risultati mega sweep PODNN/GNN"
!git push {remote_url} master